### Imports & Création des DF

In [1287]:
!gdown 1hssSUBnIR_vgZXp7JWZ21CA2zsoWFa_n -O data/dataset.zip

Downloading...
From: https://drive.google.com/uc?id=1hssSUBnIR_vgZXp7JWZ21CA2zsoWFa_n
To: /Users/matteo/Documents/Projets-hetic/Statistiques/Mercier&Associés/data/dataset.zip
100%|████████████████████████████████████████| 188k/188k [00:00<00:00, 8.71MB/s]


In [1340]:
import os, zipfile
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
import seaborn as sns
from dotenv import load_dotenv
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import ttest_ind

import plotly.io as pio
from IPython.display import display, HTML

display(HTML('<link href="https://fonts.googleapis.com/css2?family=Montserrat:wght@400;700&display=swap" rel="stylesheet">'))

load_dotenv()

True

In [1289]:
ZIP_PATH = './data/dataset.zip'
DATASET_DIR = './data'
SEED = 172
THEME = "plotly_dark"
API_KEY = os.getenv("API_KEY")

palette = px.colors.qualitative.Pastel

pio.templates[THEME].layout.font = dict(family="Montserrat", color="white")

In [1290]:
with zipfile.ZipFile(ZIP_PATH) as archive_file:
  archive_file.extractall("./data")

In [1291]:
emails = pd.read_csv(os.path.join(DATASET_DIR, "emails.csv"), sep=",")
employes = pd.read_csv(os.path.join(DATASET_DIR, "employes.csv"), sep=",")
interrogatoires = pd.read_csv(os.path.join(DATASET_DIR, "interrogatoires.csv"), sep=",")
logs_acces = pd.read_csv(os.path.join(DATASET_DIR, "logs_acces.csv"), sep=",")
preuves_materielles = pd.read_csv(os.path.join(DATASET_DIR, "preuves_materielles.csv"), sep=",")
transactions = pd.read_csv(os.path.join(DATASET_DIR, "transactions.csv"), sep=",")

### Check préalables

#### Check de transactions répétées

#### Moyennes BPM/Réponse(MS)

In [1294]:
interrogatoires

,id_suspect,num_question,type_question,texte_question,rythme_cardiaque_bpm,temps_reponse_sec
0,S01,1,neutre,Quel est votre poste exact ?,79.3,4.50
1,S01,2,neutre,Depuis quand travaillez-vous chez Mercier ?,73.5,3.59
2,S01,3,neutre,Comment décririez-vous une journée type ?,72.5,4.83
3,S01,4,neutre,Quels logiciels utilisez-vous au quotidien ?,69.7,3.90
4,S01,5,neutre,Qui sont vos collègues directs ?,66.8,4.84
...,...,...,...,...,...,...
295,S10,26,accusatrice,Comment expliquez-vous ces transferts vers les...,69.2,0.50
296,S10,27,accusatrice,Qui validait ces transactions ?,73.5,3.42
297,S10,28,accusatrice,Avez-vous effacé des fichiers récemment ?,69.1,4.29
298,S10,29,accusatrice,"Que savez-vous des 4,8 millions disparus ?",72.3,4.98


In [1295]:
df_answers = interrogatoires.merge(employes, how="outer", on="id_suspect")
df_answers = df_answers[[ 'nom', 'prenom', 'id_suspect', 'num_question', 'type_question', 'rythme_cardiaque_bpm', 'temps_reponse_sec']]
df_answers['nom'] = df_answers["nom"] + " " + df_answers['prenom']
df_answers.drop(columns="prenom", inplace=True)

df_split = df_answers

#Moyennes bpm suivant type de question
neutral_mean = (df_answers[df_answers['type_question'] == 'neutre']
         .groupby('nom')['rythme_cardiaque_bpm'].mean())
accusatory_mean = (df_answers[df_answers['type_question'] == 'accusatrice']
         .groupby('nom')['rythme_cardiaque_bpm'].mean())
bpm_mean = (df_answers.groupby('nom')['rythme_cardiaque_bpm'].mean())

#moyenne temps de réponse(ms) suivant type de question
ms_neutral_mean = (df_answers[df_answers['type_question'] == 'neutre']
         .groupby('nom')['temps_reponse_sec'].mean())
ms_accusatory_mean = (df_answers[df_answers['type_question'] == 'accusatrice']
         .groupby('nom')['temps_reponse_sec'].mean())
ms_mean = (df_answers.groupby('nom')['temps_reponse_sec'].mean())

df_answers['bpm_neutral_mean'] = df_answers['nom'].map(neutral_mean)
df_answers['bpm_accusatory_mean'] = df_answers['nom'].map(accusatory_mean)
df_answers['bpm_mean'] = df_answers['nom'].map(bpm_mean)

df_answers['ms_neutral_mean'] = df_answers['nom'].map(ms_neutral_mean)
df_answers['ms_accusatory_mean'] = df_answers['nom'].map(ms_accusatory_mean)
df_answers['ms_mean'] = df_answers['nom'].map(ms_mean)

df_answers = df_answers.drop(columns=['num_question', 'type_question', 'rythme_cardiaque_bpm', 'temps_reponse_sec'])
df_answers = df_answers[['id_suspect', 'nom', 'bpm_neutral_mean', 'bpm_accusatory_mean', 'bpm_mean', 'ms_neutral_mean', 'ms_accusatory_mean', 'ms_mean']]
df_answers = df_answers.drop_duplicates()
df_answers

,id_suspect,nom,bpm_neutral_mean,bpm_accusatory_mean,bpm_mean,ms_neutral_mean,ms_accusatory_mean,ms_mean
0,S01,Dubois Claire,71.480000,94.180000,82.830000,4.682667,9.500000,7.091333
30,S02,Moreau Antoine,69.873333,95.800000,82.836667,4.681333,8.384667,6.533000
60,S03,Bernard Sophie,68.400000,73.880000,71.140000,4.582667,4.761333,4.672000
90,S04,Haddad Karim,68.133333,93.246667,80.690000,4.720000,9.029333,6.874667
120,S05,Rousseau Élodie,73.926667,81.940000,77.933333,4.833333,6.100000,5.466667
150,S06,Lambert Pierre,71.473333,69.926667,70.700000,4.374667,5.174000,4.774333
180,S07,Petit Julie,71.433333,71.613333,71.523333,3.921333,4.516000,4.218667
210,S08,Girard Marc,70.293333,72.660000,71.476667,4.728667,5.649333,5.189000
240,S09,Martin Léa,71.280000,72.760000,72.020000,5.011333,5.360667,5.186000
270,S10,Roux Thomas,72.046667,71.840000,71.943333,4.926000,4.457333,4.691667


#### Requête API sur les mails

In [1296]:
import sys
import importlib

if './LLM' not in sys.path:
    sys.path.append('./LLM')

import agent
importlib.reload(agent)
from agent import ask_llm

print(ask_llm("renvoie les mails stp", "./data/emails.csv"))

id_envoi;id_reception;date;corps
E00001;S01;S02;2026-01-13 10:23:00;Antoine, le dossier Pégase est prêt. Procède au transfert habituel. Discrétion absolue.
E00002;S02;S01;2026-01-14 19:43:00;Claire, transfert Pégase effectué. 380k€ sur le compte habituel.
E00003;S01;S04;2026-01-18 16:15:00;Karim, les logs des comptes Hermès doivent disparaître ce soir.
E00004;S04;S01;2026-01-18 17:22:00;RE: Comptes Hermès, Logs Hermès purgés. Aucune trace résiduelle.
E00005;S01;S02;2026-01-23 09:24:00;Antoine, lance Cerbère lundi. Montant habituel. Je validerai.
E00006;S02;S01;2026-01-24 09:43:00;RE: Transfert Cerbère, Cerbère exécuté. Confirmation côté KY reçue.
E00007;S04;S02;2026-01-28 12:25:00;Pégase nettoyage, J’ai effacé l’audit trail Pégase de février. RAS.
E00008;S01;S04;2026-02-12 11:03:00;Hermès urgence, Karim, Lefèvre a posé des questions sur Hermès. Surveille ses accès.
E00009;S02;S04;2026-02-22 20:29:00;Cerbère timestamp, Karim, ajuste les timestamps Cerbère du 12 mars stp.
E00010;S04;S02;

##### -> Suspects : S01 - S02 - S04

In [1297]:
fig = go.Figure(data=[
    go.Bar(name='Question neutre', x=df_answers['id_suspect'], y=df_answers['bpm_neutral_mean'],  marker_color=palette[2]),
    go.Bar(name='Question accusatrice', x=df_answers['id_suspect'], y=df_answers['bpm_accusatory_mean'],  marker_color=palette[3])
])

fig.update_layout(barmode='group', template=THEME, title="Moyenne des BPM selon la nature de la question")
fig.show()

In [1298]:
fig = go.Figure(data=[
    go.Bar(name='Question neutre', x=df_answers['id_suspect'], y=df_answers['ms_neutral_mean'],  marker_color=palette[2]),
    go.Bar(name='Question accusatrice', x=df_answers['id_suspect'], y=df_answers['ms_accusatory_mean'],  marker_color=palette[3])
])

fig.update_layout(barmode='group', template=THEME, title="Moyenne de temps de réponse selon la nature de la question")
fig.show()

In [1299]:
print("Ecart de" , df_answers[df_answers["id_suspect"].isin(["S01", "S02", "S04"])]["bpm_accusatory_mean"].mean() - df_answers[~df_answers["id_suspect"].isin(["S01", "S02", "S04"])]["bpm_accusatory_mean"].mean(), "BPM en moyenne entre les 3 qui sortent du lot et le reste du groupe")

Ecart de 20.891746031746038 BPM en moyenne entre les 3 qui sortent du lot et le reste du groupe


##### Doutes qui convergent vers S01 / S02 / S04

In [1300]:
transactions_sum = transactions.groupby('id_suspect_initiateur')['montant_eur'].sum()

fig = px.bar(transactions_sum, title="Somme des transactions par suspect", template=THEME, color_discrete_sequence=palette,
             text_auto='.2s')
fig.show()

#### Encore un doute vers S02

### PCA & Kmeans

In [1301]:
import numpy as np
from sklearn.preprocessing import StandardScaler

features = df_answers[["bpm_neutral_mean", "bpm_accusatory_mean", "bpm_mean", "ms_neutral_mean", "ms_accusatory_mean", "ms_mean"]]
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(features),
    columns=features.columns,
    index=df_answers["nom"].values
)

In [1302]:
fig = px.imshow(X_scaled.corr(), text_auto=True, template=THEME)
fig.show()

In [1303]:
pca = PCA(n_components=0.95)
pca.fit(X_scaled)
print(pca.explained_variance_ratio_, pca.singular_values_)

[0.66942223 0.18195455 0.13478056] [6.33761264 3.3041297  2.84373585]


In [1304]:
exp_var_cumul = np.cumsum(pca.explained_variance_ratio_)

px.area(
    x=range(1, exp_var_cumul.shape[0] + 1),
    y=exp_var_cumul,
    labels={"x": "Composantes", "y": "Variance expliquée"},
    template=THEME,
    color_discrete_sequence=palette
)

##### 0.64 dès la première composante --> Jeu de données très corrélé.

#### Kmeans

In [1305]:
from sklearn.cluster import KMeans

best_k = 2

km_best = KMeans(n_clusters=best_k, random_state=SEED)
labels_best = km_best.fit_predict(X_scaled)

X_2d = pca.transform(X_scaled)[:, :2]

palette_dark = px.colors.qualitative.Set2
noms = X_scaled.index.values

fig = go.Figure()
for cluster in range(best_k):
    mask = labels_best == cluster
    fig.add_trace(go.Scatter(
        x=X_2d[mask, 0],
        y=X_2d[mask, 1],
        mode="markers+text",
        text=noms[mask],
        textposition="top center",
        marker=dict(size=10, color=palette_dark[cluster], line=dict(width=1, color="white")),
        name=f"Cluster {cluster + 1}",
    ))

fig.update_layout(
    title=f"KMeans k={best_k} (PC1 × PC2)",
    xaxis_title="PC1",
    yaxis_title="PC2",
    height=550,
    legend_title="Clusters",
    template=THEME,
)
fig.show()

Suite au clusturing on distingue bien nos 3 suspects isolés du groupe

### Modèle de détection des suspects

In [1306]:
#On créé le df_decisionTree afin de récupérer et exploiter les features intéressantes non utilisées jusqu'à maintenant
df_DecisionTree = df_answers[['id_suspect', 'nom', 'bpm_neutral_mean', 'bpm_accusatory_mean', 'ms_neutral_mean', 'ms_accusatory_mean']]
df_DecisionTree['bpm_delta'] = df_DecisionTree['bpm_accusatory_mean'] - df_DecisionTree['bpm_neutral_mean']
df_DecisionTree['ms_delta'] = df_DecisionTree['ms_accusatory_mean'] - df_DecisionTree['ms_neutral_mean']
df_DecisionTree = df_DecisionTree.drop(columns=['bpm_neutral_mean', 'bpm_accusatory_mean', 'ms_neutral_mean', 'ms_accusatory_mean'])
df_DecisionTree

/var/folders/5p/mqg45_4s6qjfqcw2nn2gkmbw0000gn/T/ipykernel_9569/910583616.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,id_suspect,nom,bpm_delta,ms_delta
0,S01,Dubois Claire,22.700000,4.817333
30,S02,Moreau Antoine,25.926667,3.703333
60,S03,Bernard Sophie,5.480000,0.178667
90,S04,Haddad Karim,25.113333,4.309333
120,S05,Rousseau Élodie,8.013333,1.266667
150,S06,Lambert Pierre,-1.546667,0.799333
180,S07,Petit Julie,0.180000,0.594667
210,S08,Girard Marc,2.366667,0.920667
240,S09,Martin Léa,1.480000,0.349333
270,S10,Roux Thomas,-0.206667,-0.468667


In [1307]:
df_DecisionTree = df_DecisionTree.merge(employes.drop(columns=["nom", "prenom", "age", "sexe", "departement", "poste", "anciennete_annees", "salaire_annuel_eur", "conges_pris_2025_jours", "performance_score"]), on="id_suspect", how="left")

df_transactions = transactions.groupby("id_suspect_initiateur")["montant_eur"].sum().reset_index()
df_transactions = df_transactions.rename(columns={"id_suspect_initiateur": "id_suspect"})
df_DecisionTree = df_DecisionTree.merge(df_transactions, on="id_suspect", how="left")

columns = ["presence_bureau_victime", "empreintes_sur_verre", "ADN_sur_porte",                                                        
               "fibres_textile_compatibles", "telephone_dans_zone"] 
df_preuves_sum = preuves_materielles
df_preuves_sum['True_amount'] = (preuves_materielles[columns] == "oui").sum(axis=1)
df_preuves_sum = df_preuves_sum[['id_suspect', 'True_amount']]
df_DecisionTree = df_DecisionTree.merge(df_preuves_sum, on="id_suspect", how="left")

In [1308]:
df_DecisionTree['SUSPICIEUX'] = df_DecisionTree["id_suspect"].isin(["S01", "S02", "S04"]).astype(int)
df_DecisionTree

,id_suspect,nom,bpm_delta,ms_delta,acces_pharmacie_societe,montant_eur,True_amount,SUSPICIEUX
0,S01,Dubois Claire,22.700000,4.817333,True,181475.64,4,1
1,S02,Moreau Antoine,25.926667,3.703333,False,1760579.34,1,1
2,S03,Bernard Sophie,5.480000,0.178667,False,109528.38,0,0
3,S04,Haddad Karim,25.113333,4.309333,False,49470.77,2,1
4,S05,Rousseau Élodie,8.013333,1.266667,False,100488.99,4,0
5,S06,Lambert Pierre,-1.546667,0.799333,False,90602.09,0,0
6,S07,Petit Julie,0.180000,0.594667,False,101521.06,0,0
7,S08,Girard Marc,2.366667,0.920667,False,800366.34,0,0
8,S09,Martin Léa,1.480000,0.349333,False,87778.64,0,0
9,S10,Roux Thomas,-0.206667,-0.468667,False,93630.81,1,0


In [ ]:
train = df_DecisionTree[['bpm_delta', 'ms_delta', 'acces_pharmacie_societe', 'montant_eur', 'True_amount']]
label = df_DecisionTree['SUSPICIEUX']

classifier = DecisionTreeClassifier(max_depth=2, random_state=SEED)
classifier.fit(train, label)

importance = pd.Series(classifier.feature_importances_, index=train.columns).sort_values(ascending=True)

fig = px.bar(importance, orientation='h', template=THEME,
               color_discrete_sequence=palette,
               title="Importance des features — Decision Tree")
fig.update_layout(showlegend=False)
fig.show()

Une seule feature suffit à l'arbre pour discriminer les coupables, le bpm_delta

In [1310]:
#vibecoding évident mais je voulais un beau graphique
import igraph as ig

feature_names = train.columns.tolist()
t = classifier.tree_
n = t.node_count

G = ig.Graph(directed=True)
G.add_vertices(n)
for i in range(n):
    if t.children_left[i] != -1:
        G.add_edge(i, t.children_left[i])
        G.add_edge(i, t.children_right[i])

lay = G.layout('rt')
M = max(lay[k][1] for k in range(n))
pos = {k: (lay[k][0], 2*M - lay[k][1]) for k in range(n)}

Xe, Ye = [], []
for e in G.es:
    s, d = e.tuple
    Xe += [pos[s][0], pos[d][0], None]
    Ye += [pos[s][1], pos[d][1], None]

node_labels, colors = [], []
for i in range(n):
    total = t.n_node_samples[i]
    is_leaf = (t.children_left[i] == -1)
    if not is_leaf:
        feat = feature_names[t.feature[i]]
        node_labels.append(f"{feat}<br>≤ {t.threshold[i]:.2f}<br>n = {total}")
        colors.append("#636EFA")
    else:
        vals = t.value[i][0]
        classe = "SUSPECT" if vals[1] > vals[0] else "INNOCENT"
        color  = "#EF553B" if classe == "SUSPECT" else "#00CC96"
        node_labels.append(f"{classe}<br>n = {total}")
        colors.append(color)

annotations = []
for e in G.es:
    s, d = e.tuple
    is_left = (t.children_left[s] == d)
    label = f"<b>≤ {t.threshold[s]:.2f}</b>" if is_left else f"<b>> {t.threshold[s]:.2f}</b>"
    annotations.append(dict(
        x=(pos[s][0] + pos[d][0]) / 2,
        y=(pos[s][1] + pos[d][1]) / 2,
        text=label, showarrow=False,
        font=dict(color="white", size=13, family="Montserrat")
    ))

fig = go.Figure()
fig.add_trace(go.Scatter(x=Xe, y=Ye, mode="lines",
    line=dict(color="white", width=2), hoverinfo="none"))
fig.add_trace(go.Scatter(
    x=[pos[k][0] for k in range(n)],
    y=[pos[k][1] for k in range(n)],
    mode="markers+text",
    marker=dict(size=90, color=colors, line=dict(width=2, color="white")),
    text=node_labels,
    textfont=dict(size=11, color="white", family="Montserrat"),
    textposition="middle center",
    hoverinfo="none"
))
fig.update_layout(
    template=THEME, showlegend=False, height=450,
    title="Decision Tree — Identification des suspects",
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    annotations=annotations
)
fig.show()

On discerne bien grâce à decisionTree nos 3 coupables grâce au delta sur les bpm entre question accusatrice et question neutre.

### Tests statistiques

#### Kruskal wallis -> Post Hoc

H0 : La question accusatrice impacte tous les suspects de la même manière
</br>H1 : La question accusatrice impacte certains suspects différemment (réaction accrue = indice de culpabilité)

In [1311]:
df_split = df_split.drop(columns=["bpm_neutral_mean", "bpm_accusatory_mean", "bpm_mean", "ms_neutral_mean", "ms_accusatory_mean", "ms_mean"])

In [1312]:
df_neutral = df_split[df_split["type_question"] == "neutre"]
df_accusatory = df_split[df_split["type_question"] == "accusatrice"]

In [1313]:
from scipy.stats import kruskal

bpm_groups = [suspect_bpm["rythme_cardiaque_bpm"].values for _, suspect_bpm in df_accusatory.groupby("nom")]
ms_groups  = [suspect_ms["temps_reponse_sec"].values     for _, suspect_ms in df_accusatory.groupby("nom")]

stat_bpm, pvalue_bpm = kruskal(*bpm_groups)
stat_ms,  pvalue_ms  = kruskal(*ms_groups)

pd.DataFrame([
    {"Métrique": "BPM",              "H": round(stat_bpm, 4), "p-value": round(pvalue_bpm, 25)},
    {"Métrique": "Temps de réponse", "H": round(stat_ms,  4), "p-value": round(pvalue_ms,  25)},
])

,Métrique,H,p-value
0,BPM,101.2431,8.817367e-18
1,Temps de réponse,68.4802,3.021860e-11


Si p < 0.05 → H0 rejetée. Donc on invalide l'Hypothèse 0 ce qui induit que certains suspects sont impactés différemment par la question accusatrice

In [1314]:
from scikit_posthocs import posthoc_dunn                                                                                                
                                                                                                                                          
dunn_bpm = posthoc_dunn(df_accusatory, val_col="rythme_cardiaque_bpm", group_col="nom", p_adjust="bonferroni",)                          
fig = px.imshow(dunn_bpm)
fig.update_layout(
    template=THEME,
)
fig.show()

#### Welch t-test

On va tester avec le test de Welch si les vrais criminels ont plus laissé de preuves que le reste du groupe

H0 : On trouve un nombre de preuves équitable pour tous les suspects
</br>H1 : On trouve un nombre de preuves + important pour les suspects

In [1350]:
suspicious = df_DecisionTree[df_DecisionTree["SUSPICIEUX"] == 1]["True_amount"]
innocents  = df_DecisionTree[df_DecisionTree["SUSPICIEUX"] == 0]["True_amount"]

t, p = ttest_ind(suspicious, innocents, equal_var=False)
print(f"t={t:.3f}, p={p:.4f}")

if p < 0.05:
    print(f"H0 rejetée — les suspects ont significativement plus de preuves (p={p:.4f})")
else:
    print(f"H0 non rejetée — pas de différence significative du nombre de preuves (p={p:.4f})")

t=1.545, p=0.2014
H0 non rejetée — pas de différence significative du nombre de preuves (p=0.2014)
